In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error 
import xgboost as xgb
from pathlib import Path

data_path = Path.cwd().parent / 'data' / 'car_fuel_efficiency_2026.csv'
df = pd.read_csv(data_path)



In [ ]:
# 1. Valores faltantes con ceros
df = df.fillna(0)

# 2. División 
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

# 3. Identificar la variable objetivo
y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']

# 4. DictVectorizer (one-hot - categóricas)
train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

1. Original splitting feature

In [16]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



2. RMSE on validation

In [17]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
rmse = root_mean_squared_error(y_val, y_pred)
print(round(rmse, 3))

1.833


In [ ]:
print(df.shape)   # original
print(X_train.shape, X_val.shape)
print(df_train.shape, df_val.shape)

(10000, 11)
(6000, 16) (2000, 16)
(6000, 10) (2000, 10)


3. Number of estimators

In [18]:
for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)
    print(n, round(root_mean_squared_error(y_val, y_pred), 3))

10 1.833
50 1.774
100 1.77
150 1.769


4. Best max depth

In [19]:
results = {}
for d in [10, 15, 20, 25]:
    rmses = []
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(n_estimators=n, max_depth=d,
                                   random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)
        rmses.append(root_mean_squared_error(y_val, y_pred))
    results[d] = np.mean(rmses)

print(results)
print('Mejor max_depth:', min(results, key=results.get))

{10: np.float64(1.7543924943801008), 15: np.float64(1.7842243445385957), 20: np.float64(1.7867711878395158), 25: np.float64(1.7863054936840876)}
Mejor max_depth: 10


5. Most important feature

In [20]:
rf = RandomForestRegressor(n_estimators=10, max_depth=20,
                           random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

importances = pd.DataFrame({
    'feature': dv.get_feature_names_out(),
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print(importances.head(10))

                         feature  importance
9                     model_year    0.296107
15                vehicle_weight    0.203313
6             fuel_type=Gasoline    0.160712
8                     horsepower    0.078271
4            engine_displacement    0.061281
0                   acceleration    0.054998
14                    origin=USA    0.042152
2   drivetrain=Front-wheel drive    0.020062
11                     num_doors    0.019401
5               fuel_type=Diesel    0.016571


6. XGBoost eta 

In [21]:
features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)

watchlist = [(dtrain, 'train'), (dval, 'val')]

def train_xgb(eta):
    params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,
        'objective': 'reg:squarederror',
        'nthread': 8,
        'seed': 1,
        'verbosity': 1,
    }
    evals_result = {}
    xgb.train(params, dtrain, num_boost_round=100,
              evals=watchlist, evals_result=evals_result, verbose_eval=10)
    return evals_result['val']['rmse']

rmse_03 = train_xgb(0.3)
rmse_01 = train_xgb(0.1)

print('eta=0.3 -> mejor:', min(rmse_03), ' final:', rmse_03[-1])
print('eta=0.1 -> mejor:', min(rmse_01), ' final:', rmse_01[-1])

[0]	train-rmse:2.40653	val-rmse:2.46782
[10]	train-rmse:1.47125	val-rmse:1.73205
[20]	train-rmse:1.39368	val-rmse:1.73766
[30]	train-rmse:1.33243	val-rmse:1.74548
[40]	train-rmse:1.25326	val-rmse:1.76296
[50]	train-rmse:1.18645	val-rmse:1.77517
[60]	train-rmse:1.10265	val-rmse:1.79401
[70]	train-rmse:1.04604	val-rmse:1.80444
[80]	train-rmse:0.99487	val-rmse:1.81308
[90]	train-rmse:0.94986	val-rmse:1.82135
[99]	train-rmse:0.90106	val-rmse:1.82646
[0]	train-rmse:2.74739	val-rmse:2.78635
[10]	train-rmse:1.80592	val-rmse:1.93152
[20]	train-rmse:1.55847	val-rmse:1.75076
[30]	train-rmse:1.47544	val-rmse:1.71667
[40]	train-rmse:1.43074	val-rmse:1.71289
[50]	train-rmse:1.40368	val-rmse:1.71358
[60]	train-rmse:1.37884	val-rmse:1.71688
[70]	train-rmse:1.36283	val-rmse:1.71919
[80]	train-rmse:1.34578	val-rmse:1.72100
[90]	train-rmse:1.32546	val-rmse:1.72303
[99]	train-rmse:1.31192	val-rmse:1.72481
eta=0.3 -> mejor: 1.7320478064805087  final: 1.8264579949460302
eta=0.1 -> mejor: 1.7121714763059768